# Agenting banking system


In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [5]:
from langchain.agents import create_agent
from langchain.tools import tool

model = "gpt-4o-mini"

## Single agent

In [10]:
system_message = """You are a helpful banking agent. Decide whether to use the tools available to you to answer the user's question. If you use a tool, provide the input to the tool in your response. If you do not use a tool, provide a direct answer to the user's question. Do not make up information. If you do not know the answer, say "I don't know"."""

In [7]:
@tool
def get_account_balance(account_id: str) -> str:
    """
    Get the account balance for a given account number.
    """
    # In a real application, you would query your database or an API to get the account balance.
    # Here, we will return a mock balance for demonstration purposes.
    mock_balances = {
        "123456": "$1,000.00",
        "654321": "$2,500.00",
        "111222": "$3,750.00"
    }
    return mock_balances.get(account_id, "Account not found.")

@tool
def get_transaction_history(account_id: str) -> str:
    """
    Get the transaction history for a given account number.
    """
    # In a real application, you would query your database or an API to get the transaction history.
    # Here, we will return a mock transaction history for demonstration purposes.
    mock_transactions = {
        "123456": [
            {"date": "2024-01-01", "amount": "-$50.00", "description": "Grocery Store"},
            {"date": "2024-01-02", "amount": "-$20.00", "description": "Gas Station"},
            {"date": "2024-01-03", "amount": "+$1,000.00", "description": "Paycheck"}
        ],
        "654321": [
            {"date": "2024-01-01", "amount": "-$100.00", "description": "Electronics Store"},
            {"date": "2024-01-02", "amount": "-$30.00", "description": "Restaurant"},
            {"date": "2024-01-03", "amount": "+$2,500.00", "description": "Paycheck"}
        ],
        "111222": [
            {"date": "2024-01-01", "amount": "-$200.00", "description": "Furniture Store"},
            {"date": "2024-01-02", "amount": "-$40.00", "description": "Coffee Shop"},
            {"date": "2024-01-03", "amount": "+$3,750.00", "description": "Paycheck"}
        ]
    }
    transactions = mock_transactions.get(account_id)
    if transactions is None:
        return "Account not found."
    
    transaction_history = "\n".join([f"{t['date']}: {t['amount']} - {t['description']}" for t in transactions])
    return transaction_history

@tool
def get_current_address(account_id: str) -> str:
    """
    Get the current address for a given account number.
    """
    # In a real application, you would query your database or an API to get the current address.
    # Here, we will return a mock address for demonstration purposes.
    mock_addresses = {
        "123456": "123 Main St, Anytown, USA",
        "654321": "456 Elm St, Othertown, USA",
        "111222": "789 Oak St, Sometown, USA"
    }
    return mock_addresses.get(account_id, "Account not found.")

In [8]:
tools = [get_account_balance, get_transaction_history, get_current_address]

In [14]:
banking_agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=system_message,
)


In [16]:
result = banking_agent.invoke({"messages": [{"role": "user", "content": "Give me the account balance for account number 123456."}]})

In [20]:
result['messages'][-1].content

'The account balance for account number 123456 is $1,000.00.'

## Multi-agent

In [25]:
system_prompt_accounts_subagent = """You are a helpful banking agent. Decide whether to use the tools available to you to answer the user's question. If you use a tool, provide the input to the tool in your response. If you do not use a tool, provide a direct answer to the user's question. Do not make up information. If you do not know the answer, say "I don't know"."""

tools_accounts_subagent = [get_account_balance]

accounts_subagent = create_agent(
    model=model,
    tools=tools_accounts_subagent,
    system_prompt=system_prompt_accounts_subagent
)

@tool("accounts_subagent", description="provide information about a user's account, such as balance.")
def call_accounts_subagent(query: str):
    result = accounts_subagent.invoke({"messages": [{"role": "user", "content": query}]})
    return result["messages"][-1].content

In [26]:
system_prompt_transactions_subagent = """You are a helpful banking agent. Decide whether to use the tools available to you to answer the user's question. If you use a tool, provide the input to the tool in your response. If you do not use a tool, provide a direct answer to the user's question. Do not make up information. If you do not know the answer, say "I don't know"."""

tools_transactions_subagent = [get_transaction_history]

transactions_subagent = create_agent(
    model=model,
    tools=tools_transactions_subagent,
    system_prompt=system_prompt_transactions_subagent
)

@tool("transactions_subagent", description="provide information about a user's transactions, such as transaction history.")
def call_transactions_subagent(query: str):
    result = transactions_subagent.invoke({"messages": [{"role": "user", "content": query}]})
    return result["messages"][-1].content

In [27]:
system_prompt_services_subagent = """You are a helpful banking agent. Decide whether to use the tools available to you to answer the user's question. If you use a tool, provide the input to the tool in your response. If you do not use a tool, provide a direct answer to the user's question. Do not make up information. If you do not know the answer, say "I don't know"."""

tools_services_subagent = [get_current_address]

services_subagent = create_agent(
    model=model,
    tools=tools_services_subagent,
    system_prompt=system_prompt_services_subagent
)

@tool("services_subagent", description="provide information about a user's services, such as current address.")
def call_services_subagent(query: str):
    result = services_subagent.invoke({"messages": [{"role": "user", "content": query}]})
    return result["messages"][-1].content


In [33]:
system_prompt_coordinator_agent = """You are a helpful banking agent. Decide which sub-agent to use to answer the user's question. 
If you use a sub-agent, provide the input to the sub-agent in your response. If you do not use a sub-agent, 
provide a direct answer to the user's question. Do not make up information. If you do not know the answer, say "I don't know".
Available sub-agents:
- accounts_subagent: provide information about a user's account, such as balance.
- transactions_subagent: provide information about a user's transactions, such as transaction history.
- services_subagent: provide information about a user's services, such as current address.
"""

coordinator_agent = create_agent(
    model=model,
    tools=[call_accounts_subagent, call_transactions_subagent, call_services_subagent],
    system_prompt=system_prompt_coordinator_agent
)

In [30]:
result = coordinator_agent.invoke({"messages": [{"role": "user", "content": "Give me the account balance for account number 123456 and give me the current address ."}]})

In [31]:
result['messages'][-1].content

'The account balance for account number 123456 is $1,000.00. The current address for this account is 123 Main St, Anytown, USA.'

In [32]:
result

{'messages': [HumanMessage(content='Give me the account balance for account number 123456 and give me the current address .', additional_kwargs={}, response_metadata={}, id='139199f4-46dc-4266-8e25-0f321f160802'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 54, 'prompt_tokens': 196, 'total_tokens': 250, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_edb0948d4f', 'id': 'chatcmpl-EQkfcrojG8wQrcjuNOvXYZj9AksjN', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0c6f9-7e02-7a60-8526-9f61a6cf02dc-0', tool_calls=[{'name': 'accounts_subagent